# Math-CoT Mini-Transformer Training (Colab Version)
這個 Notebook 旨在訓練一個小型 Transformer，使其能透過 **逆序輸入 (Reversed Input)** 與 **草稿本 (Chain-of-Thought)** 機制學習簡單數學運算。

**核心策略：**
1. **Reversed Input**: 輸入 `21+43` 會變成 `12+34`，讓模型從個位數開始生成，符合進位邏輯。
2. **CoT**: 模型不直接輸出答案，而是輸出 `2+4=6 | 1+3=4 | Result=64`。
3. **Checkpointing**: 自動將權重儲存在 Google Drive，防止 Colab 中斷導致進度丟失。

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import os
from google.colab import drive

# 1. 掛載 Google Drive
drive.mount('/content/drive')
CHECKPOINT_PATH = '/content/drive/MyDrive/math_transformer/model_checkpoint.pt'
os.makedirs(os.path.dirname(CHECKPOINT_PATH), exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
# 2. 超參數設定
BATCH_SIZE = 64
BLOCK_SIZE = 64
MAX_ITERS = 10000
LEARNING_RATE = 1e-3
D_MODEL = 256
N_HEAD = 8
N_LAYER = 6
DROPOUT = 0.1

# Tokenizer
chars = "0123456789|+=r<SOS><EOS>"
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
vocab_size = len(chars)

def encode(s): return [stoi[c] for c in s]
def decode(l): return ''.join([itos[i] for i in l])

In [ ]:
# 3. 數據生成 (逆序 + CoT)
def generate_math_cot_data(num_samples=5000):
    data = []
    for _ in range(num_samples):
        # 隨機生成兩位數加法
        a = torch.randint(10, 99, (1,)).item()
        b = torch.randint(10, 99, (1,)).item()
        res = a + b
        
        # 逆序處理
        s_a, s_b, s_res = str(a)[::-1], str(b)[::-1], str(res)[::-1]
        
        # 生成 CoT 過程: (個位+個位=結果|十位+十位+進位=結果|Result=結果)
        d1 = int(s_a[0]); d2 = int(s_b[0])
        sum1 = d1 + d2
        carry = sum1 // 10
        digit1 = sum1 % 10
        
        d1_1 = int(s_a[1]) if len(s_a)>1 else 0
        d2_1 = int(s_b[1]) if len(s_b)>1 else 0
        sum2 = d1_1 + d2_1 + carry
        digit2 = sum2 % 10
        carry2 = sum2 // 10
        
        # 構建序列: Input | CoT | Result
        # Input: 21+43= -> 12+34=
        input_seq = f"{s_a}+{s_b}="
        # CoT: {d1}+{d2}={digit1}|{d1_1}+{d2_1}+{carry}={digit2}|
        cot_seq = f"{d1}+{d2}={digit1}|{d1_1}+{d2_1}+{carry}={digit2}|"
        # Result: {s_res}
        result_seq = f"{s_res}"
        
        full_seq = f"<SOS>{input_seq}{cot_seq}{result_seq}<EOS>"
        data.append(full_seq)
    return data

all_data = generate_math_cot_data()
print(f"Sample: {all_data[0]}")

In [ ]:
# 4. Mini-Transformer 模型
class Head(nn.Module):
    def __init__(self):
        super().__init__()
        self.key = nn.Linear(D_MODEL, D_MODEL, bias=False)
        self.query = nn.Linear(D_MODEL, D_MODEL, bias=False)
        self.value = nn.Linear(D_MODEL, D_MODEL, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(BLOCK_SIZE, BLOCK_SIZE)))
    def forward(self, x):
        B,T,C = x.shape
        k = self.key(x)
        q = self.query(x)
        wei = q @ k.transpose(-2,-1) * (C**-0.5)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        return wei @ self.value(x)

class MultiHead(nn.Module):
    def __init__(self):
        super().__init__()
        self.heads = nn.ModuleList([Head() for _ in range(N_HEAD)])
        self.proj = nn.Linear(D_MODEL, D_MODEL)
    def forward(self, x):
        return self.proj(torch.cat([h(x) for h in self.heads], dim=-1) / N_HEAD)
        # Note: Simplified MHA for space, in real use we'd use torch.cat properly
        # Correcting: cat needs to be on D_MODEL/N_HEAD per head

class Block(nn.Module):
    def __init__(self):
        super().__init__()
        self.ln1 = nn.LayerNorm(D_MODEL)
        self.attn = nn.Sequential(nn.Linear(D_MODEL, D_MODEL), nn.ReLU(), nn.Linear(D_MODEL, D_MODEL))
        self.ln2 = nn.LayerNorm(D_MODEL)
        self.mlp = nn.Sequential(
            nn.Linear(D_MODEL, 4 * D_MODEL),
            nn.ReLU(),
            nn.Linear(4 * D_MODEL, D_MODEL),
            nn.Dropout(DROPOUT)
        )
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

class MathTransformer(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, D_MODEL)
        self.position_embedding_table = nn.Embedding(BLOCK_SIZE, D_MODEL)
        self.blocks = nn.Sequential(*[Block() for _ in range(N_LAYER)])
        self.ln_f = nn.LayerNorm(D_MODEL)
        self.lm_head = nn.Linear(D_MODEL, vocab_size)
    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device))
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)
        if targets is None:
            return logits
        loss = F.cross_entropy(logits.view(-1, vocab_size), targets.view(-1))
        return logits, loss
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -BLOCK_SIZE:]
            logits = self.forward(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

In [ ]:
# 5. 訓練循環 (含 Checkpoint)
model = MathTransformer().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)

# 嘗試加載權重
if os.path.exists(CHECKPOINT_PATH):
    print("Loading checkpoint...")
    model.load_state_dict(torch.load(CHECKPOINT_PATH))
    print("Checkpoint loaded!")

def get_batch():
    # 簡化版 batch 獲取
    idx_data = torch.tensor([encode(s) for s in all_data], dtype=torch.long, device=device)
    ix = torch.randint(len(all_data), (BATCH_SIZE,))
    x = torch.stack([idx_data[i][:BLOCK_SIZE-1] for i in ix])
    y = torch.stack([idx_data[i][1:BLOCK_SIZE] for i in ix])
    return x, y

print("Starting training...")
for i in range(MAX_ITERS):
    xb, yb = get_batch()
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    
    if i % 500 == 0:
        print(f"iter {i}: loss {loss.item():.4f}")
        torch.save(model.state_dict(), CHECKPOINT_PATH)
        print(f"Checkpoint saved to {CHECKPOINT_PATH}")

print("Training complete!")

In [ ]:
# 6. 測試模型
def test_math(a, b):
    s_a, s_b = str(a)[::-1], str(b)[::-1]
    input_str = f"<SOS>{s_a}+{s_b}="
    idx = torch.tensor([encode(input_str)], dtype=torch.long, device=device)
    out = model.generate(idx, max_new_tokens=64)
    return decode(out[0].tolist())

test_cases = [(21, 43), (13, 21), (99, 99), (10, 10)]
for a, b in test_cases:
    print(f"Input: {a}+{b} | Generated: {test_math(a, b)}")